# 01 - Turn every frame into numbers (DINOv2 embeddings)

**Why:** a classifier cannot read JPEG pixels directly. A pretrained network (DINOv2) already "understands" images,
so we pass each frame through it and keep its output vector (an *embedding*, 1536 numbers per frame).
Similar-looking frames get similar vectors, so a change between frames = a change in vectors.

**Steps:** find data -> load DINOv2 -> for every clip, embed all 30 frames -> save as `.npy` files.

**Output:** `dinov2b_train.npy` (3000, 30, 1536) and `dinov2b_test.npy` (1750, 30, 1536), saved as float16 to save space.

**Settings on Kaggle:** Accelerator = GPU T4, Internet = On (downloads the model once), competition `cooked-or-not` attached.
**Local smoke test:** set env var `SMOKE=50` to use only 50 clips per split.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel

# Use the GPU if there is one (Kaggle T4 or your RTX 3050), otherwise the slow CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU found)")

## 1. Find the data and decide where to save
The same notebook runs on Kaggle and on your PC, so we search for `train_labels.csv` in both places.
`SMOKE` is a quick-test switch: it keeps only the first N clips so a local test takes under a minute.

In [ ]:
# Kaggle path first, then the local data/ folder (notebooks start inside kaggle/<step>/)
found = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True) + glob.glob("../../data/train_labels.csv")
assert found, "train_labels.csv not found - attach the competition / check the path"
ROOT = os.path.dirname(found[0])

SMOKE = int(os.environ.get("SMOKE", 0)) or None       # e.g. SMOKE=50 -> only 50 clips per split

# On Kaggle, anything saved in /kaggle/working is downloadable afterwards
on_kaggle = os.path.exists("/kaggle/working")
OUT = "/kaggle/working/features" if on_kaggle else ("../../artifacts/features_smoke" if SMOKE else "../../artifacts/features")
os.makedirs(OUT, exist_ok=True)

# Clip ids in the SAME order as train_labels.csv / sample_submission.csv (so row i of the features = row i of the csv)
train_ids = pd.read_csv(f"{ROOT}/train_labels.csv")["clip_id"].tolist()[:SMOKE]
test_ids = pd.read_csv(f"{ROOT}/sample_submission.csv")["clip_id"].tolist()[:SMOKE]
print(ROOT, "| train clips:", len(train_ids), "| test clips:", len(test_ids))

## 2. Load the DINOv2 model
We only *use* the model (no training), so `.eval()` switches off training behaviour, and `.half()` uses 16-bit numbers on the GPU: faster and half the memory.
The image size must be a multiple of 14 (DINOv2 cuts the image into 14x14 pixel patches): 224 = 16x14 and 392 = 28x14. Our frames are 384x224, so we stretch the width slightly to 392.

In [ ]:
MODEL = "facebook/dinov2-base"          # needs Internet = On the first time
model = AutoModel.from_pretrained(MODEL).eval().to(device)
if device == "cuda":
    model = model.half()

H, W = 224, 392
# The network was trained with images normalised by these numbers, so we must do the same
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

## 3. Read one clip = 30 frames
A `Dataset` just says "give me item number i". Ours reads the 30 JPEGs of clip i in the right order (frame_000 ... frame_029)
and returns them as one tensor of shape (30, 3, H, W).

In [ ]:
class ClipDataset(torch.utils.data.Dataset):
    def __init__(self, split, ids):
        self.split = split          # "train" or "test"
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        folder = f"{ROOT}/{self.split}/{self.ids[i]}"
        frames = []
        for j in range(30):
            img = Image.open(f"{folder}/frame_{j:03d}.jpg").convert("RGB").resize((W, H), Image.BICUBIC)
            frames.append(np.asarray(img))
        frames = np.stack(frames)                                   # (30, H, W, 3)
        return torch.from_numpy(frames).permute(0, 3, 1, 2)         # (30, 3, H, W), uint8

## 4. Embed all clips
For each batch of clips: flatten (clips x 30 frames) into one big batch of images, normalise, run DINOv2, then reshape back to (clips, 30, features).
Each frame's vector = **CLS token** (DINOv2's summary of the whole image, 768 numbers) joined with the **average of the patch tokens**
(the average over all image regions, another 768 numbers) = 1536 numbers.

In [ ]:
@torch.no_grad()                      # we are not training, so no gradients (saves memory)
def embed_split(split, ids, batch_clips=4):
    # Windows cannot use DataLoader worker processes inside notebooks, so 0 workers there
    loader = torch.utils.data.DataLoader(
        ClipDataset(split, ids), batch_size=batch_clips, shuffle=False,   # shuffle=False keeps the csv order
        num_workers=0 if os.name == "nt" else 2,
    )
    results = []
    for clips in tqdm(loader, desc=split):
        n = clips.shape[0]                                          # clips in this batch
        x = clips.flatten(0, 1).to(device).float() / 255           # (n*30, 3, H, W) values in 0..1
        x = (x - mean) / std                                        # normalise like DINOv2 expects
        if device == "cuda":
            x = x.half()
        tokens = model(pixel_values=x).last_hidden_state            # (n*30, 1+patches, 768)
        vec = torch.cat([tokens[:, 0], tokens[:, 1:].mean(1)], dim=-1)   # CLS + mean patch -> (n*30, 1536)
        results.append(vec.view(n, 30, -1).float().cpu().numpy().astype(np.float16))
    return np.concatenate(results)                                  # (num_clips, 30, 1536)

## 5. Run on train and test, then save
The printed shapes must be (3000, 30, 1536) and (1750, 30, 1536) on the full run, with 0 NaNs.

In [ ]:
t0 = time.time()
train_feats = embed_split("train", train_ids)
test_feats = embed_split("test", test_ids)

np.save(f"{OUT}/dinov2b_train.npy", train_feats)
np.save(f"{OUT}/dinov2b_test.npy", test_feats)
pd.Series(train_ids, name="clip_id").to_csv(f"{OUT}/train_ids.csv", index=False)   # row order record
pd.Series(test_ids, name="clip_id").to_csv(f"{OUT}/test_ids.csv", index=False)

print(train_feats.shape, test_feats.shape, f"{time.time() - t0:.0f}s", "NaNs:", np.isnan(train_feats).sum(), np.isnan(test_feats).sum())

## Next (local)
Download: `kaggle kernels output aryanbhendarkar/cooked-or-not-01-extract-features -p artifacts/features`
Then: pool the 30 frames into one vector per clip (`pool_features` in `src/common.py`), train logistic regression with `make_folds()`, read the CV AUC.